In [58]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

category_cache = project_root / "data/processed/lab06"

encoder = joblib.load(
    category_cache / "category_encoder.joblib"
)

train_ids = np.load(
    category_cache / "X_cat_train.npy",
    mmap_mode="r",
)

columns = encoder.feature_names_in_.tolist()
known_counts = [len(values) for values in encoder.categories_]

mask_ids = torch.tensor(known_counts, dtype=torch.long) + 1
vocabulary_sizes = (mask_ids + 1).tolist()

pd.DataFrame({
    "column": columns,
    "known_categories": known_counts,
    "unknown_id": 0,
    "mask_id": mask_ids.tolist(),
    "embedding_table_size": vocabulary_sizes,
})

,column,known_categories,unknown_id,mask_id,embedding_table_size
0,payment_currency,15,0,16,17
1,receiving_currency,15,0,16,17
2,payment_format,6,0,7,8


In [59]:
def mask_one_category(category_ids, mask_ids, *, generator):
    rows = torch.arange(len(category_ids))

    # Randomly select one column for each transaction.
    masked_columns = torch.randint(
        category_ids.shape[1],
        (len(category_ids),),
        generator=generator,
    )

    # Prediction classes start at zero; our known input IDs start at one.
    targets = category_ids[rows, masked_columns] - 1

    # Preserve the original inputs.
    masked_ids = category_ids.clone()
    masked_ids[rows, masked_columns] = mask_ids[masked_columns]

    return masked_ids, masked_columns, targets

In [60]:
original = torch.from_numpy(
    np.array(train_ids[:8], copy=True)
)
original_snapshot = original.clone()

generator = torch.Generator().manual_seed(42)

masked, masked_columns, targets = mask_one_category(
    original,
    mask_ids,
    generator=generator,
)

assert torch.equal(original, original_snapshot)
assert ((masked != original).sum(dim=1) == 1).all()

rows = torch.arange(len(original))

torch.testing.assert_close(
    targets + 1,
    original[rows, masked_columns],
)

print("Original inputs preserved.")
print("Exactly one category hidden per transaction.")
print("Targets match the hidden values.")

Original inputs preserved.
Exactly one category hidden per transaction.
Targets match the hidden values.


In [61]:
original_values = pd.DataFrame(
    encoder.inverse_transform(original.numpy() - 1),
    columns=columns,
)

inspection = original_values.copy()

for row, column in enumerate(masked_columns.tolist()):
    inspection.iat[row, column] = "[MASK]"

inspection["predict_column"] = [
    columns[column]
    for column in masked_columns.tolist()
]

inspection["correct_value"] = [
    original_values.iat[row, column]
    for row, column in enumerate(masked_columns.tolist())
]

inspection

,payment_currency,receiving_currency,payment_format,predict_column,correct_value
0,[MASK],US Dollar,ACH,payment_currency,US Dollar
1,US Dollar,US Dollar,[MASK],payment_format,Credit Card
2,US Dollar,[MASK],Credit Card,receiving_currency,US Dollar
3,US Dollar,[MASK],Credit Card,receiving_currency,US Dollar
4,[MASK],US Dollar,ACH,payment_currency,US Dollar
5,US Dollar,US Dollar,[MASK],payment_format,Credit Card
6,US Dollar,[MASK],Cheque,receiving_currency,US Dollar
7,US Dollar,US Dollar,[MASK],payment_format,Credit Card


In [62]:
cache_dir = project_root / "data/processed/lab08"
cache_dir.mkdir(parents=True, exist_ok=True)

masking_config = {
    "categorical_columns": columns,
    "known_counts": known_counts,
    "mask_ids": mask_ids.tolist(),
    "vocabulary_sizes": vocabulary_sizes,
    "labeled_fraction": 0.1,
    "seed": 42,
}

(cache_dir / "masking_config.json").write_text(
    json.dumps(masking_config, indent=2) + "\n"
)

288

## Step 2 — Pre-training and fine-tuning

**Goal:** compare two identical laundering models with the same 10% label budget. One starts randomly; the other starts with embeddings and a transformer block learned from masked categories.

**Time:** about 65 minutes, including reading and running the code.

Restart the kernel, then run the new Step 2 cells below in order. This section loads its own saved inputs. Training stays on CPU with two threads.

### A. Prepare the label budget and inputs

Pre-training will see all training-period category rows and no laundering labels. For supervised training, we simulate limited labels by selecting 10% of training rows with stratified sampling. Both models use the same saved subset.

The sample uses label values to preserve prevalence; it is a controlled simulation, not a claim that real-world label availability is random. Validation remains the same September 7–8 set. Final-test data stays reserved.

In [63]:
import json
import sys
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.embeddings import predict_embedding_scores
from src.neural_training import train_one_epoch
from src.pretraining import MaskedCategoryModel, pretrain_one_epoch
from src.tabular_transformer import TabularTransformer

torch.set_num_threads(2)

cache_dir = project_root / "data/processed/lab08"
previous_cache = project_root / "data/processed/lab05"
category_cache = project_root / "data/processed/lab06"
reports_dir = project_root / "reports"
models_dir = project_root / "models"
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

config = json.loads((cache_dir / "masking_config.json").read_text())
seed = config["seed"]

encoder = joblib.load(category_cache / "category_encoder.joblib")
preprocessor = joblib.load(previous_cache / "preprocessor.joblib")
n_numeric = preprocessor.output_indices_["categorical"].start

assert n_numeric == 8
assert encoder.feature_names_in_.tolist() == config["categorical_columns"]
assert [len(c) for c in encoder.categories_] == config["known_counts"]
assert config["mask_ids"] == [k + 1 for k in config["known_counts"]]
assert config["vocabulary_sizes"] == [k + 2 for k in config["known_counts"]]

train_categories = torch.from_numpy(
    np.load(category_cache / "X_cat_train.npy")
)
train_labels = np.load(previous_cache / "y_train.npy")
assert len(train_categories) == len(train_labels)


In [64]:
labeled_indices, _ = train_test_split(
    np.arange(len(train_labels)),
    train_size=config["labeled_fraction"],
    stratify=train_labels,
    random_state=seed,
)
labeled_indices = np.sort(labeled_indices)

np.save(cache_dir / "labeled_indices.npy", labeled_indices)
train_source_rows = np.load(previous_cache / "train_source_rows.npy")
np.save(
    cache_dir / "labeled_source_rows.npy",
    train_source_rows[labeled_indices],
)

# Copy only the selected numerical rows for supervised training.
train_matrix = np.load(
    previous_cache / "X_train.npy", mmap_mode="r"
)
labeled_numeric = torch.from_numpy(np.array(
    train_matrix[labeled_indices, :n_numeric],
    dtype=np.float32,
    order="C",
    copy=True,
))
labeled_categories = train_categories[
    torch.from_numpy(labeled_indices)
]
y_labeled = torch.from_numpy(train_labels[labeled_indices])

val_matrix = np.load(
    previous_cache / "X_validation.npy", mmap_mode="r"
)
val_numeric = torch.from_numpy(np.array(
    val_matrix[:, :n_numeric],
    dtype=np.float32,
    order="C",
    copy=True,
))
val_categories = torch.from_numpy(
    np.load(category_cache / "X_cat_validation.npy")
)
y_validation = np.load(previous_cache / "y_validation.npy")

assert len(val_numeric) == len(val_categories) == len(y_validation)
assert torch.unique(y_labeled).tolist() == [0.0, 1.0]

print(f"Rows available for pre-training: {len(train_categories):,}")
print(f"Rows with supervised labels: {len(y_labeled):,}")
print(f"Positive supervised labels: {int(y_labeled.sum()):,}")
print(f"Supervised laundering rate: {y_labeled.mean().item():.3%}")

Rows available for pre-training: 1,379,551
Rows with supervised labels: 137,955
Positive supervised labels: 180
Supervised laundering rate: 0.130%


Expected: **1,379,551 pre-training rows; 137,955 labeled rows; 180 positive labels.**

Saving both subset positions and source-row IDs lets us reuse and audit exactly which labels were available. The numerical array in RAM contains only the labeled training rows; pre-training needs only the categorical inputs.

### B. Pre-train by reconstructing hidden categories

Open `src/pretraining.py` and look at:
- `MaskedCategoryModel`: a shared categorical encoder plus three reconstruction heads, predicting 15, 15, and 6 known values.
- `pretrain_one_epoch()`: masks one category per row and computes cross-entropy only at hidden positions.
- The backbone's laundering classifier is unused during this stage and receives no gradients.

No sigmoid is applied before multiclass cross-entropy. Each reconstruction target is a known category's zero-based index.

The table reports **training reconstruction metrics**, not laundering detection or held-out accuracy. Masking and batch shuffling have separate generators, both created once before the epoch loop. Their state advances across epochs.

In [65]:
pretraining_config = {
    "n_numeric": n_numeric,
    "known_counts": tuple(config["known_counts"]),
    "token_dim": 8,
    "num_heads": 2,
    "feedforward_dim": 32,
}

torch.manual_seed(seed)
pretrainer = MaskedCategoryModel(**pretraining_config)

pretrain_loader = DataLoader(
    TensorDataset(train_categories),
    batch_size=4096,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(seed),
)
mask_generator = torch.Generator().manual_seed(seed)
pretrain_optimizer = torch.optim.AdamW(
    pretrainer.parameters(), lr=1e-3, weight_decay=1e-4
)

pretraining_epochs = 3
pretraining_records = []

for epoch in range(1, pretraining_epochs + 1):
    start = perf_counter()
    metrics = pretrain_one_epoch(
        pretrainer,
        pretrain_loader,
        pretrain_optimizer,
        generator=mask_generator,
    )
    result = {
        "epoch": epoch,
        **metrics,
        "seconds": perf_counter() - start,
    }
    pretraining_records.append(result)
    print(
        f"Pre-training {epoch}/{pretraining_epochs} | "
        f"loss={metrics['loss']:.4f} | "
        f"masked accuracy={metrics['accuracy']:.2%} | "
        f"{result['seconds']:.1f}s",
        flush=True,
    )

pretraining_history = pd.DataFrame(pretraining_records).rename(
    columns={
        f"accuracy_{i}": f"accuracy_{name}"
        for i, name in enumerate(config["categorical_columns"])
    }
)
pretraining_history.to_csv(
    reports_dir / "lab_08_pretraining.csv", index=False
)

torch.save(
    {
        "model_state": pretrainer.state_dict(),
        "model_config": pretraining_config,
        "epochs": pretraining_epochs,
        "seed": seed,
        "categorical_columns": config["categorical_columns"],
    },
    models_dir / "lab08_pretraining.pt",
)

pretraining_history


Pre-training 1/3 | loss=1.3005 | masked accuracy=56.23% | 5.7s
Pre-training 2/3 | loss=0.5520 | masked accuracy=79.38% | 5.0s
Pre-training 3/3 | loss=0.5093 | masked accuracy=79.99% | 4.8s


,epoch,loss,accuracy,accuracy_payment_currency,accuracy_receiving_currency,accuracy_payment_format,seconds
0,1,1.300525,0.562350,0.677202,0.614971,0.395467,5.676259
1,2,0.551950,0.793784,0.978898,0.972564,0.429160,5.022485
2,3,0.509268,0.799852,0.983985,0.984088,0.431670,4.783364


### C. Train the two laundering models

Both models have the same architecture, including the extra MASK entries, and use the same 137,955 labeled transactions.

For the pretrained model, these two operations copy learned weights:
```python
model.embeddings.load_state_dict(pretrained.backbone.embeddings.state_dict())
model.context.load_state_dict(pretrained.backbone.context.state_dict())
```

The reconstruction heads are not transferred. The laundering classifier starts fresh and identically in both runs. All supervised model parameters remain trainable: this is full fine-tuning, not a frozen feature extractor.

A new optimizer is created for each supervised run. It starts with fresh optimization state for the laundering objective. Loading state dictionaries copies values; the two models do not share mutable parameter objects.

Both runs get five supervised epochs and the same batch order. Pre-training adds extra compute, so this compares equal label and supervised-update budgets, not equal total compute. We will include that cost in Step 3.

In [66]:
# Load the saved pre-training result so transfer uses an explicit checkpoint.
pretraining_checkpoint = torch.load(
    models_dir / "lab08_pretraining.pt",
    map_location="cpu",
    weights_only=True,
)
assert (
    pretraining_checkpoint["categorical_columns"]
    == config["categorical_columns"]
)
pretrained = MaskedCategoryModel(
    **pretraining_checkpoint["model_config"]
)
pretrained.load_state_dict(pretraining_checkpoint["model_state"])

supervised_config = {
    "n_numeric": n_numeric,
    "vocabulary_sizes": tuple(config["vocabulary_sizes"]),
    "positive_rate": y_labeled.mean().item(),
    "token_dim": 8,
    "num_heads": 2,
    "feedforward_dim": 32,
    "use_transformer": True,
}

finetuning_epochs = 5
finetuning_records = []

for run_name in ["scratch", "pretrained_finetuned"]:
    torch.manual_seed(seed)
    model = TabularTransformer(**supervised_config)

    if run_name == "pretrained_finetuned":
        model.embeddings.load_state_dict(
            pretrained.backbone.embeddings.state_dict()
        )
        model.context.load_state_dict(
            pretrained.backbone.context.state_dict()
        )

    assert all(p.requires_grad for p in model.parameters())

    labeled_loader = DataLoader(
        TensorDataset(labeled_numeric, labeled_categories, y_labeled),
        batch_size=4096,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(seed),
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(1, finetuning_epochs + 1):
        start = perf_counter()
        train_loss = train_one_epoch(
            model, labeled_loader, optimizer, loss_fn
        )
        validation_scores = predict_embedding_scores(
            model, val_numeric, val_categories
        )
        validation_ap = average_precision_score(
            y_validation, validation_scores
        )
        result = {
            "model": run_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_ap": validation_ap,
            "seconds": perf_counter() - start,
        }
        finetuning_records.append(result)
        print(
            f"{run_name} {epoch}/{finetuning_epochs} | "
            f"loss={train_loss:.5f} | "
            f"validation AP={validation_ap:.4f} | "
            f"{result['seconds']:.1f}s",
            flush=True,
        )

    np.save(
        cache_dir / f"{run_name}_validation_scores.npy",
        validation_scores,
    )
    torch.save(
        {
            "model_state": model.state_dict(),
            "model_config": supervised_config,
            "epochs": finetuning_epochs,
            "seed": seed,
            "labeled_fraction": config["labeled_fraction"],
            "labeled_rows": len(y_labeled),
            "positive_labels": int(y_labeled.sum()),
            "initialization": run_name,
            "pretraining_epochs": (
                pretraining_checkpoint["epochs"]
                if run_name == "pretrained_finetuned" else 0
            ),
            "categorical_columns": config["categorical_columns"],
        },
        models_dir / f"lab08_{run_name}.pt",
    )

finetuning_history = pd.DataFrame(finetuning_records)
finetuning_history.to_csv(
    reports_dir / "lab_08_finetuning.csv", index=False
)

finetuning_history


scratch 1/5 | loss=0.00978 | validation AP=0.0169 | 1.5s
scratch 2/5 | loss=0.00875 | validation AP=0.0145 | 1.4s
scratch 3/5 | loss=0.00766 | validation AP=0.0188 | 1.6s
scratch 4/5 | loss=0.00744 | validation AP=0.0201 | 1.6s
scratch 5/5 | loss=0.00736 | validation AP=0.0227 | 1.4s
pretrained_finetuned 1/5 | loss=0.00961 | validation AP=0.0104 | 1.3s
pretrained_finetuned 2/5 | loss=0.00839 | validation AP=0.0150 | 1.4s
pretrained_finetuned 3/5 | loss=0.00785 | validation AP=0.0138 | 1.3s
pretrained_finetuned 4/5 | loss=0.00766 | validation AP=0.0181 | 1.4s
pretrained_finetuned 5/5 | loss=0.00758 | validation AP=0.0179 | 1.5s


,model,epoch,train_loss,validation_ap,seconds
0,scratch,1,0.009775,0.016858,1.524215
1,scratch,2,0.008751,0.014536,1.373584
2,scratch,3,0.007665,0.018847,1.586350
3,scratch,4,0.007438,0.020058,1.565786
4,scratch,5,0.007360,0.022713,1.437651
5,pretrained_finetuned,1,0.009614,0.010445,1.333831
6,pretrained_finetuned,2,0.008390,0.015013,1.402377
7,pretrained_finetuned,3,0.007848,0.013844,1.328012
8,pretrained_finetuned,4,0.007663,0.018067,1.418376
9,pretrained_finetuned,5,0.007583,0.017937,1.491276


### Done condition

You should have:
- Three rows in `reports/lab_08_pretraining.csv`.
- Ten rows in `reports/lab_08_finetuning.csv`.
- A pre-training checkpoint, two supervised checkpoints, and two validation score files.
- Saved indices and source-row IDs for the shared labeled subset.

Bring the two history tables back to the chat. Step 3 will compare AP, detections at 50 reviews per day, and total training cost. Reconstruction accuracy alone does not establish useful transfer: the two currency fields match in about 98.4% of training rows.

**Learning to retain:** objectives can change between stages; encoder weights can transfer while task heads change; a new optimizer resets training state; copying weights differs from sharing model objects.

References: [TabTransformer pre-training](https://arxiv.org/html/2012.06678v1), [PyTorch cross-entropy](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html).

## Step 3 — Evaluate transfer and close the lab

**Question:** did learning to reconstruct categories help us detect laundering with fewer labels?

**Time:** about 20 minutes. No more training is needed. Run the evaluation cell below, read the interpretation, then review the lab report.

### A. Compare the two models on the actual task

Load the saved final-epoch scores and evaluate the same September 7–8 transactions at **50 reviews per day**. First check that score positions correspond to the saved validation rows and labels.

The two measures answer different questions:
- **Average precision (AP):** how well are positives ranked across the full validation set? It summarizes precision across recall levels; it is not the fraction of cases caught.
- **Detections at the review budget:** how many positives occur in the 50 highest-scored transactions each day?

Precision at budget divides detections by reviews; recall at budget divides detections by all validation positives. Here the denominators are 100 and 1,036.


In [67]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import evaluate_scores

processed_dir = project_root / "data/processed"
reports_dir = project_root / "reports"
validation = pd.read_parquet(
    processed_dir / "lab02_validation.parquet",
    columns=["source_row", "timestamp", "is_laundering"],
)
np.testing.assert_array_equal(
    validation["source_row"],
    np.load(processed_dir / "lab05/validation_source_rows.npy"),
)
np.testing.assert_array_equal(
    validation["is_laundering"],
    np.load(processed_dir / "lab05/y_validation.npy"),
)

pretraining_history = pd.read_csv(reports_dir / "lab_08_pretraining.csv")
finetuning_history = pd.read_csv(reports_dir / "lab_08_finetuning.csv")
comparison_records = []

for run_name in ["scratch", "pretrained_finetuned"]:
    scores = np.load(
        processed_dir / "lab08" / f"{run_name}_validation_scores.npy"
    )
    metrics = evaluate_scores(validation, scores, daily_budget=50)
    run_history = finetuning_history.loc[
        finetuning_history["model"].eq(run_name)
    ].sort_values("epoch")
    np.testing.assert_allclose(
        metrics["average_precision"], run_history.iloc[-1]["validation_ap"]
    )
    supervised_seconds = run_history["seconds"].sum()
    pretraining_seconds = (
        pretraining_history["seconds"].sum()
        if run_name == "pretrained_finetuned" else 0.0
    )
    comparison_records.append({
        "model": run_name,
        **metrics,
        "supervised_seconds": supervised_seconds,
        "pretraining_seconds": pretraining_seconds,
        "total_seconds": supervised_seconds + pretraining_seconds,
    })

comparison = pd.DataFrame(comparison_records)
comparison.to_csv(reports_dir / "lab_08_model_comparison.csv", index=False)
comparison.round(4)


,model,average_precision,alerts,laundering_caught,precision_at_budget,recall_at_budget,supervised_seconds,pretraining_seconds,total_seconds
0,scratch,0.0227,100,4,0.04,0.0039,7.4876,0.0000,7.4876
1,pretrained_finetuned,0.0179,100,2,0.02,0.0019,6.9739,15.4821,22.4560


### B. Interpret the result and retain the lesson

Your saved runs give:

| Measure | Scratch | Pretrained + fine-tuned |
|---|---:|---:|
| Average precision | 0.0227 | 0.0179 |
| Laundering caught / 100 reviews | 4 | 2 |
| Precision at budget | 4% | 2% |
| Recall at budget | 0.39% | 0.19% |
| Recorded total time | 7.5 s | 23.8 s |

For example, scratch recall is `4 / 1036`, while its precision at budget is `4 / 100`. Both AP and detections favor scratch in this run.

**What did pre-training learn?** Its final online training accuracy was about 98.4% for each hidden currency and 43.2% for hidden payment format. Because the currency columns match in about 98.4% of training rows, copying the visible currency is an easy baseline. The result is consistent with learning this relationship, but accuracy alone does not prove the model used that strategy. These are training reconstruction metrics, not held-out laundering metrics.

**Decision:** this pre-training setup did not help our downstream task. Extra training on a different objective does not guarantee better detection. The easy currency relationship is a plausible explanation, not an established cause. One seed, one label subset, three pre-training epochs, and five supervised epochs cannot establish that pre-training generally fails.

The scratch and pretrained runs have equal supervised data and update budgets. Pre-training adds 16.7 seconds. Supervised timings include training, validation prediction, and AP calculation each epoch; pre-training timings include masking, training, and reconstruction metrics. Input preparation and checkpoint saving are excluded.

Keep boosting with history as the project's existing reference. Its earlier result used all training labels, so it is not a matched baseline for this 10%-label experiment. We evaluate the planned final epoch, rather than selecting whichever epoch looks best afterward. The final test remains reserved.

**Read `reports/lab_08.md`**, especially its short learning-takeaways section. The comparison CSV belongs in `reports/` because it summarizes an experiment; scores and sampled row IDs remain under `data/processed/lab08/`.

**Done condition:** you can explain why 98.4% currency reconstruction accuracy can coexist with worse laundering AP, and identify exactly which weights transferred to the supervised model. No need to tune more models to finish this lab.
